In [1]:
# app.py
from dash import Dash, dcc, html, Input, Output, callback
import pandas as pd
import numpy as np
import ibis
from ibis import _
from dash import State, no_update
import json
import dash_bootstrap_components as dbc
from dash import ctx


import datetime
from dotenv import load_dotenv
import os

import pdb

from dash import dash_table


In [2]:
load_dotenv()
kwargs_con = {
    'host':os.getenv('db_host_'),
    'user':os.getenv('db_user_'),
    'port':os.getenv('db_port_'),
    'password':os.getenv('db_password_'),
    'database':os.getenv('db_database_')
}

kwargs_tbl_name = {
    'tbl_pr_':os.getenv('db_table_province_'),
    'tbl_car_':os.getenv('db_table_car_'),
    'tbl_sp_':os.getenv('db_table_sp_')
}

class IbisPostgresConnection:
    """Context Manager برای مدیریت اتصال Ibis به PostgreSQL"""
    
    def __init__(self, connection_params):
        """
        Initialize connection manager
        
        Args:
            connection_params: پارامترهای اتصال به دیتابیس
        """
        self.connection_params = connection_params
        self.connection= None
    
    def __enter__(self):
        """باز کردن اتصال هنگام ورود به context"""
        try:
            self.connection = ibis.postgres.connect(**self.connection_params)
            # print('--- Connected ---')
            return self.connection
        except Exception as e:
            raise ConnectionError(f"خطا در اتصال به دیتابیس: {e}")
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """بستن اتصال هنگام خروج از context"""
        if self.connection:
            try:
                self.connection.disconnect()
                # print('--- Disconnected ---')
            except Exception as close_error:
                print(f"⚠️  خطا در بستن اتصال: {close_error}")
            finally:
                self.connection = None
        
        # اگر خطایی در context رخ داده بود، آن را suppress نکن
        # return False = “خطا را به بیرون منتقل کن (propagate)”
        # return True = “خطا را سرکوب کن (suppress)”
        #برای connection managerها معمولاً return False مناسب‌تر است چون می‌خواهیم خطاهای واقعی برنامه دیده شوند.
        if exc_type is not None:
            print(f"خطای {exc_type.__name__}: {exc_val}")
        
            # مشاهده traceback
            import traceback
            traceback.print_tb(exc_tb)
        return False
        

def load_provinces():
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.select('provinceName').distinct().execute()
        distinct_pr = distinct_pr.provinceName.to_list()
    return distinct_pr

def load_cities(prs):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.filter(distinct_pr.provinceName.isin(prs))
        distinct_pr = distinct_pr.select('cityName').distinct().execute()
        distinct_pr = distinct_pr.cityName.to_list()
    return distinct_pr

def load_roadNames(prs,cities):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
                      )
        distinct_pr = distinct_pr.select('sppRoadName').distinct().execute()
        distinct_pr = distinct_pr.sppRoadName
    return distinct_pr


def load_cameraIds(prs,cities,roadNames):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (
            distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
            .filter(distinct_pr.sppRoadName.isin(roadNames))
                      )
        distinct_pr = distinct_pr.select("cameraId").distinct().execute()
        distinct_pr = distinct_pr.cameraId
    return distinct_pr

def load_data(prs,cities,roadNames,cameraIds):
    with IbisPostgresConnection(kwargs_con) as con:
        tbl = con.table(kwargs_tbl_name['tbl_pr_'])
        data = (
            tbl.filter(_.provinceName.isin(prs))
            .filter(_.cityName.isin(cities))
            .filter(_.sppRoadName.isin(roadNames))
            .filter(_.cameraId.isin(cameraIds))
        ).execute()
    return data



app = Dash(__name__ ,
    external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dcc.Tabs([
        dcc.Tab(label='استان', children=[
            dcc.Dropdown(id='Id_province',multi=True)
        ]),
        dcc.Tab(label='شهرستان', children=[
            dcc.Dropdown(id='Id_city',multi=True)
        ]),
        dcc.Tab(label='محور', children=[
            dcc.Dropdown(id='Id_roadName',multi=True)
        ]),
        dcc.Tab(label='آیدی دوربین', children=[
            dcc.Dropdown(id='Id_cameraId',multi=True)
        ])
    ]),
   
    dash_table.DataTable(id="id_tbl" ,export_format="csv",    export_headers="display"),
    
])

@callback(
    Output('Id_province', 'options'),
    Output('Id_province', 'value'),
    Input('Id_province', 'id')
)
def init_provinces(_):
    df = load_provinces()
    options = [{'label': r, 'value': r} for r in df]
    value = options[0]['value'] if options else None
    return options, value


@callback(
    Output("Id_city", "options"),
    Output("Id_city", "value"),
    Input("Id_province", "value")
)
def init_city(prs):
    if not prs:
        return [], []
    ###### This line is very important
    if isinstance(prs, str):
        prs = [prs]
    ######
    df = load_cities(prs)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    
    return options, value

@callback(
    Output('Id_roadName', 'options'),
    Output('Id_roadName', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value')    
)
def init_roadName(prs,cities):
    if not prs or not cities:
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]

    df = load_roadNames(prs, cities)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    return options, value

@callback(
    Output('Id_cameraId', 'options'),
    Output('Id_cameraId', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value'),
    Input('Id_roadName', 'value')
)
def init_cameraId(prs,cities,roadNames):
    if not prs or not cities or not roadNames :
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    df = load_cameraIds(prs, cities,roadNames)    
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]    
    return options, value


@app.callback(
    Output("id_tbl", "columns"),
    Output("id_tbl", "data"),
    Input("Id_province","value"),
    Input("Id_city","value"),
    Input("Id_roadName","value"),
    Input("Id_cameraId","value")
)
def init_data(prs,cities,roadNames,cameraIds):
    if not cameraIds:
        return [], []
    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    if isinstance(cameraIds, str):
        roadNames = [cameraIds]

    df = load_data(prs,cities,roadNames,cameraIds)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")


    
if __name__ == '__main__':
    app.run(debug=True,port=9871)